<a href="https://colab.research.google.com/github/AjaySagar9/Research-Agent/blob/main/ai_research_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install ddgs
!pip install newspaper3k
!pip install beautifulsoup4
!pip install lxml_html_clean
!pip install fpdf

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

class CustomSummarizer:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        # Automatically move model to GPU if available
        self.device = 0 if torch.cuda.is_available() else -1
        if self.device == 0:
            self.model.to("cuda")

    def __call__(self, text, max_length=150, min_length=50, **kwargs):
        # Prepare inputs, moving them to the correct device
        inputs = self.tokenizer([text], max_length=1024, return_tensors="pt", truncation=True)
        if self.device == 0:
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        summary_ids = self.model.generate(
            inputs["input_ids"], num_beams=4, max_length=max_length, min_length=min_length, early_stopping=True
        )
        # Move output back to CPU for decoding if necessary
        decoded_summary = self.tokenizer.decode(summary_ids[0].cpu(), skip_special_tokens=True)
        return [{"summary_text": decoded_summary}]

summarizer = CustomSummarizer("facebook/bart-large-cnn")

print("✅ BART Loaded (custom pipeline)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

✅ BART Loaded (custom pipeline)


In [ ]:
from ddgs import DDGS

def search_web(query):

    links = []

    with DDGS() as ddgs:

        results = ddgs.text(
            query,
            max_results=5
        )

        for r in results:
            links.append(r)

    return links

In [ ]:
from newspaper import Article

def extract_content(url):

    try:

        article = Article(url)

        article.download()

        article.parse()

        return article.text

    except:

        return ""

In [ ]:
def collect_research(topic):

    results = search_web(topic)

    research_text = ""

    for item in results:

        url = item["href"]

        print("Reading:", url)

        content = extract_content(url)

        research_text += content[:3000]

    return research_text

In [ ]:
def summarize_text(text):

    chunks = []

    for i in range(
        0,
        len(text),
        1000
    ):

        chunks.append(
            text[i:i+1000]
        )

    summaries = []

    for chunk in chunks[:5]:

        try:

            result = summarizer(
                chunk,
                max_length=150,
                min_length=50,
                do_sample=False
            )

            summaries.append(
                result[0]["summary_text"]
            )

        except:
            pass

    return "\n".join(
        summaries
    )

In [ ]:
def research_agent(topic):

    print("🔍 Searching...")

    data = collect_research(
        topic
    )

    print("📝 Summarizing...")

    summary = summarize_text(
        data
    )

    return summary

In [ ]:
topic = input(
    "Research Topic: "
)

report = research_agent(
    topic
)

print(report)

Research Topic: Ai in Healthcare
🔍 Searching...
Reading: https://en.wikipedia.org/wiki/AI_in_healthcare
Reading: https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare
Reading: https://grokipedia.com/page/AI_in_Healthcare_Specialization
Reading: https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/
Reading: https://ep.jhu.edu/news/ai-in-healthcare-applications-and-impact/
📝 Summarizing...
Artificial intelligence in healthcare refers to the application of artificial intelligence (AI) to analyze and understand complex medical and healthcare data. It can often augment and in some cases exceed human capabilities by providing better or faster ways to diagnose, treat, or prevent disease. Research is ongoing into its applications across various medical subdisciplines and related industries.
resents unprecedented ethical concerns related to issues such as data privacy, automation of jobs, and amplifying already existing algorithmic bias. New technologies such as AI are often met with 

In [ ]:
with open(
    "research_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(report)

print("✅ TXT Saved")

✅ TXT Saved


In [ ]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

# The error occurs because fpdf 1.7.2 doesn't support UTF-8 by default.
# We must replace or encode characters that fall outside the Latin-1 range.
safe_report = (
    report
    .replace("\u201c", '"')  # Left smart quote
    .replace("\u201d", '"')  # Right smart quote
    .replace("\u2018", "'")  # Left single quote
    .replace("\u2019", "'")  # Right single quote
    .replace("\u2014", "-")  # Em dash
    .replace("\u2013", "-")  # En dash
    .encode('latin-1', 'replace').decode('latin-1') # Final safety check
)

pdf.multi_cell(
    0,
    10,
    "Research Report\n\n" + safe_report
)

pdf.output("research_report.pdf")

print("✅ PDF Created Successfully")

✅ PDF Created Successfully


In [ ]:
from google.colab import files

files.download(
    "research_report.pdf"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>